# Riemannian Objects, Projections, And Smooth Images

This notebook demonstrates the higher-level object operations available in the Riemannian layer.

In [ ]:
from pathlib import Path
import sys
import math

import matplotlib.pyplot as plt
import numpy as np

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'pyproject.toml').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from geo import (
    EuclideanNeighborhood,
    EuclideanPlaneSpace,
    FloatPoint,
    FloatVector,
    Hyperplane,
    ManifoldChart,
    RealLineSpace,
    RiemannianGeometricObject,
)


In [ ]:
plane = EuclideanPlaneSpace()
upper = plane.half_plane((0.0, 1.0), name='upper')
right = plane.half_plane((1.0, 0.0), name='right')
quadrant = upper & right

print('quadrant contains (1, 1):', FloatPoint(1.0, 1.0) in quadrant)
print('quadrant contains (-1, 1):', FloatPoint(-1.0, 1.0) in quadrant)

In [ ]:
source_line = RiemannianGeometricObject.from_charted(
    plane,
    Hyperplane((0.0, 1.0), offset=1.0),
)
source_half_line = source_line & plane.half_plane((1.0, 0.0), offset=0.0)
target_line = Hyperplane((0.0, 1.0), offset=0.0)

parallel_projection = source_half_line.project_along_direction_onto(
    Hyperplane((0.0, 1.0), offset=1.0),
    target_line,
    (0.0, -1.0),
)
central_projection = source_half_line.project_from_point_onto(
    Hyperplane((0.0, 1.0), offset=1.0),
    target_line,
    FloatPoint(0.0, 2.0),
)

print('parallel projection contains (1, 0):', FloatPoint(1.0, 0.0) in parallel_projection)
print('central projection contains (2, 0):', FloatPoint(2.0, 0.0) in central_projection)

In [ ]:
source_space = RealLineSpace()
source = source_space.subset((0.0, 2.0), name='segment')

def target_chart(point):
    center = FloatPoint(point)
    return ManifoldChart(
        lambda candidate: FloatPoint(candidate) - center,
        lambda coordinates: center + FloatVector(coordinates),
        dim=2,
        domain_contains=plane.contains,
        image=EuclideanNeighborhood.whole(2),
    )


parabola = source.image_under_smooth_map(
    lambda point: FloatPoint(point, point * point),
    lambda point: float(FloatPoint(point)[0]),
    plane,
    target_chart,
    contains_image_point=lambda point: (
        0.0 <= FloatPoint(point)[0] <= 2.0 and
        math.isclose(
            FloatPoint(point)[1],
            FloatPoint(point)[0] * FloatPoint(point)[0],
            rel_tol=1e-9,
            abs_tol=1e-9,
        )
    ),
)

print('parabola contains (1, 1):', FloatPoint(1.0, 1.0) in parabola)
print('parabola contains (1, 0):', FloatPoint(1.0, 0.0) in parabola)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x = np.linspace(0.0, 2.0, 200)
ax.plot(x, np.ones_like(x), label='source half-line on y=1', linewidth=2)
ax.plot(x, np.zeros_like(x), label='parallel projection', linewidth=2)

x_central = np.linspace(0.0, 4.0, 200)
ax.plot(x_central, np.zeros_like(x_central), '--', label='central projection image')

x_parabola = np.linspace(0.0, 2.0, 200)
ax.plot(x_parabola, x_parabola**2, label='smooth image parabola', linewidth=2)

ax.scatter([0.0], [2.0], color='black', label='projection center')
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-0.5, 4.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.25)
ax.legend()
ax.set_title('Examples of object images and projections')